# Analýza používania zdielaných práčiek a sušičiek na kolejách 17. Listopadu

Na kolejách sa nachádza **18 práčiek a 6 sušičiek** od spoločnosti AppWash. 

Informácie o týchto práčkach sa dajú získať pomocou AppWash **pythonovskej knižnice**, ktorú som použil pre kontinuálne sledovanie stavov práčok a šušičiek.

Vytvoril som si skript, ktorý v minútových intervaloch pre každý z 24 strojov zistil či je zrovna obsadený, prípadne od kedy je v prevádzke.
Tento skript som nechal bežať od **18. mája do 17. júna**, a za tento čas vytvoril asi milión riadkové CSV.

Následne som dáta premenil do formátu zoznamu **eventov**, teda pre každé pranie/sušenie je riadok s údajmi o stroji a začiatkom a koncom cyklu.

Tieto dáta sú v priloženom súbore `17_listopad_machine_events.csv`, a ceľkovo to je **4159** praní/sušení.

## Ciele

V projekte chcem spraviť exploratívnu analýzu dát, a následne otestovať hypotézu, že práčky a sušičky sú viac vyťažené cez víkendy, ako cez týždeň.

## Načítanie dát

A orezanie na celé týždne, nakoľko sú počty praní dosť závislé na dni v týždni. (Čo bude vudno aj v testovaní hypotézy.)

Z dát taktiež vymažeme 

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px


In [2]:
UTC_OFFSET_HOURS = 2

df = pd.read_csv("./17_listopad_machine_events.csv")
df["start_time"] = pd.to_datetime(df["start_time"], unit="s") + pd.Timedelta(hours=UTC_OFFSET_HOURS)
df["end_time"] = pd.to_datetime(df["end_time"]).dt.round("s") + pd.Timedelta(hours=UTC_OFFSET_HOURS)

START = df["start_time"].min()
END = df["end_time"].max()
PERIOD = pd.Timedelta(days=28)

df = df.sort_values("start_time").reset_index(drop=True)
df = df[df["start_time"] <= START + PERIOD]
df


,service_id,physical_location,type,name,start_time,end_time
0,50274,B 2nd Floor,DRYER,dryer,2026-05-18 10:04:03,2026-05-18 11:21:03
1,50242,B 5th Floor,WASHING_MACHINE,washing machine,2026-05-18 10:22:53,2026-05-18 11:24:04
2,50272,B 2nd Floor,WASHING_MACHINE,washing machine,2026-05-18 10:23:10,2026-05-18 11:21:03
3,46853,A 10th Floor,WASHING_MACHINE,washing machine,2026-05-18 10:25:23,2026-05-18 11:21:03
4,46854,A 10th Floor,WASHING_MACHINE,washing machine,2026-05-18 10:25:40,2026-05-18 11:21:03
...,...,...,...,...,...,...
3868,46824,B 10th Floor,WASHING_MACHINE,washing machine,2026-06-15 09:11:42,2026-06-15 10:02:58
3869,46825,B 10th Floor,WASHING_MACHINE,washing machine,2026-06-15 09:11:47,2026-06-15 09:13:42
3870,46823,B 10th Floor,WASHING_MACHINE,washing machine,2026-06-15 09:12:10,2026-06-15 10:01:58
3871,46825,B 10th Floor,WASHING_MACHINE,washing machine,2026-06-15 09:12:43,2026-06-15 10:01:58


In [3]:
df["cycle_duration_minutes"] = (df["end_time"] - df["start_time"]).dt.total_seconds() / 60
df


,service_id,physical_location,type,name,start_time,end_time,cycle_duration_minutes
0,50274,B 2nd Floor,DRYER,dryer,2026-05-18 10:04:03,2026-05-18 11:21:03,77.000000
1,50242,B 5th Floor,WASHING_MACHINE,washing machine,2026-05-18 10:22:53,2026-05-18 11:24:04,61.183333
2,50272,B 2nd Floor,WASHING_MACHINE,washing machine,2026-05-18 10:23:10,2026-05-18 11:21:03,57.883333
3,46853,A 10th Floor,WASHING_MACHINE,washing machine,2026-05-18 10:25:23,2026-05-18 11:21:03,55.666667
4,46854,A 10th Floor,WASHING_MACHINE,washing machine,2026-05-18 10:25:40,2026-05-18 11:21:03,55.383333
...,...,...,...,...,...,...,...
3868,46824,B 10th Floor,WASHING_MACHINE,washing machine,2026-06-15 09:11:42,2026-06-15 10:02:58,51.266667
3869,46825,B 10th Floor,WASHING_MACHINE,washing machine,2026-06-15 09:11:47,2026-06-15 09:13:42,1.916667
3870,46823,B 10th Floor,WASHING_MACHINE,washing machine,2026-06-15 09:12:10,2026-06-15 10:01:58,49.800000
3871,46825,B 10th Floor,WASHING_MACHINE,washing machine,2026-06-15 09:12:43,2026-06-15 10:01:58,49.250000


### Analýza dĺžky cyklov a čistenie dát

In [4]:
fig5 = px.histogram(df, x="cycle_duration_minutes", color="type", 
                    title="Freq. of washes and dries by day of week",
                    category_orders={"day_of_week": ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]},
                    color_discrete_map={"WASHING_MACHINE": "blue", "DRYER": "orange"},
                    nbins=200
)
fig5.update_layout(xaxis_title="Day of Week", yaxis_title="Count")
fig5.show()

fig6 = px.histogram(df, x="cycle_duration_minutes", color="type", 
                    title="Freq. of washes and dries by day of week",
                    category_orders={"day_of_week": ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]},
                    color_discrete_map={"WASHING_MACHINE": "blue", "DRYER": "orange"},
)
fig6.update_layout(xaxis_title="Day of Week", yaxis_title="Count")
fig6.show()


V dátach je veľa ultra krátkych cyklov, okolo 1 minúty, po ktorých okamžite nasleduje cyklus s iným začiatkom na rovnakej práčke, jedná sa o artefakt v dátach, predpokladám že je to spôsobené tým že keď človek stlačí START na práčke predtým ako zaplatí cez aplikáciu tak sa už práčka uzamkne a pravdepodobne zmení stav na OCCUPIED, ale následne sa cyklus spustí až po zaplatení.

Preto tieto hodnoty vyfiltrujem.

Okrem toho v dátach vidíme docela dosť dlhších, ale stále podozrivo krátkych cyklov, predpokladám že sa jedná o chyby práčok/sušičiek, z vlastnej skúšenosti sa mi raz začas proste pranie/sušenie preruší a práčka/sušička vypíše chybovú hlášku.

Taktiež si môžeme všimnúť že cykly niekdy trvajú výrazne dlhšie ako inokedy, na sušičke sa to zhoduje s praxou, nakoľko sušičky sušia až kým neprestanú detekovať vlhkosť, u práčok si to z vlastnej skúsenosti vysvetliť neviem, ale môže sa jednať o pomalšie ECO cykly.

In [5]:
df = df[df["cycle_duration_minutes"] > 3]
df = df.sort_values("start_time").reset_index(drop=True)


## Exploratívna analýza

In [10]:
print("=== ANALYSYS ===")
print(f"Number of events: {len(df)}")
print(f"Number of washes: {len(df[df['type'] == 'WASHING_MACHINE'])}")
print(f"Median cycle duration: {df['cycle_duration_minutes'].median():.2f} minutes")
print(f"Number of dries: {len(df[df['type'] == 'DRYER'])}")
print(f"Median cycle duration: {df[df['type'] == 'DRYER']['cycle_duration_minutes'].median():.2f} minutes")



# Kumulativny graf poctu prani a suseni

df_washes = df[df["type"] == "WASHING_MACHINE"].reset_index(drop=True)
df_dryers = df[df["type"] == "DRYER"].reset_index(drop=True)

fig1 = px.line(df_washes, x="start_time", y=df_washes.index, title="Cumulative number of washes over time")
fig1.update_layout(xaxis_title="Time", yaxis_title="Cumulative number of washes")
fig1.show()

fig2 = px.line(df_dryers, x="start_time", y=df_dryers.index, title="Cumulative number of dries over time", color_discrete_sequence=["orange"])
fig2.update_layout(xaxis_title="Time", yaxis_title="Cumulative number of dries")
fig2.show()


=== ANALYSYS ===
Number of events: 2994
Number of washes: 2412
Median cycle duration: 50.55 minutes
Number of dries: 582
Median cycle duration: 43.54 minutes


Je zaujímavé že počty sušení sú výrazne nižšie ako počty praní, predpokladám že to je spôsobené tým že sušičky majú väčšiu kapacitu (do jednej sa zmestia aj dve práčky), a taktiež veľa ľudí suší bez nich, na sušiaku.

Vidíme že počty praní a sušení sa vyvíjajú lineárne, čo by sme aj štandardne čakali, avšak dá sa čakať že viac ku koncu skúškového sa budú zmenšovať. Nakoľko budú ľuďia odchádzať domov. na prázdniny

### Frekvencie prania a sušenia vzhľadom na deň v týždni/čas

In [7]:
df["day_of_week"] = df["start_time"].dt.day_name()
df["hour_of_day"] = df["start_time"].dt.hour

fig3 = px.histogram(df, x="day_of_week", color="type", 
                    title="Freq. of washes and dries by day of week", 
                    category_orders={"day_of_week": ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]},
                    color_discrete_map={"WASHING_MACHINE": "blue", "DRYER": "orange"},
)
fig3.update_layout(xaxis_title="Day of Week", yaxis_title="Count")
fig3.show()

fig4 = px.histogram(df, x="hour_of_day", color="type", title="Freq. of washes and dries by hour of day", color_discrete_map={"WASHING_MACHINE": "blue", "DRYER": "orange"})
fig4.update_layout(xaxis_title="Hour of Day", yaxis_title="Count")
fig4.show()


Z dát vidíme že počas víkendu ľudia viac perú a sušia, túto hypotézu overím aj pomocou permutačnej hypotézy.

Taktiež vidíme že ľudia viac perú poobede a večer, ako ráno a v noci.

## Testovanie hypotézy

**Hypotéza: Cez víkend je vyššia frekvencia prania ako cez pracovné dni**
+ $H_0$ - frekvencie sú rovnaké

Hypotézu budem testovať pomocou **testu dobrej zhody**.

Ak platí $H_0$, tak sa pravdepodobnosti kategórií pre každý deň rovnajú $p_i=\frac{1}{7}$.

Následne dosadíme do vzťahu pre test dobrej zhody, a porovnáme hodnotu $\chi ^{2}$ s tabuľkou pri hladine $\alpha=0.05$.

$${\displaystyle \chi ^{2}=\sum _{i=1}^{k}{\frac {(X_{i}-Np_{i})^{2}}{Np_{i}}}}$$

In [14]:
p = np.array([1/7] * 7)
N = df.shape[0]
X = df.groupby("day_of_week").size().reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]).values
print(f"{N=}")
print(f"{X=}")

chi2 = ((X - N*p)**2 / (N*p)).sum()
print(f"Chi-squared: {chi2:.4f}")


N=2994
X=array([387, 428, 410, 408, 400, 456, 505])
Chi-squared: 23.1496


Získanú hodnotu Chí Kvadrát teraz vieme porovnať s tabuľkou pre Chí Kvadrát test s **6** stupňami voľnosti (počet kategórií - 1).

| $df$ | 0.995 | 0.99 | 0.975 | 0.95 | 0.90 | 0.10 | 0.05 | 0.025 | 0.01 | 0.005 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| **6** | 0.676 | 0.872 | 1.237 | 1.635 | 2.204 | 10.645 | 12.592 | 14.449 | 16.812 | 18.548 |

Vidíme že $p<0.005$, a teda zamietame nulovú hypotézu.